# Bato: trek risk model evaluation

Makes every figure and table for the report from the `training_sample` table.
The code is in `backend/app/ml/` (our decision tree and random forest are in `tree.py` and `forest.py`).

**Kernel:** choose the Python in `backend/.venv` (run `cd backend && uv sync` first). PostgreSQL must be running and seeded (`uv run python -m app.seed`).

In [ ]:
import os, sys
from pathlib import Path

BACKEND = Path.cwd().parent / 'backend' if Path.cwd().name == 'ml' else Path.cwd() / 'backend'
os.chdir(BACKEND)              # reads backend/.env (DATABASE_URL)
sys.path.insert(0, str(BACKEND))

from IPython.display import Image, Markdown, display
from app.ml import evaluate

results = evaluate.run()
display(Markdown((evaluate.OUT_DIR / 'comparison.md').read_text()))

## Training data
Agency ratings + survey answers + rule-generated rows (rules in `backend/app/ml/dataset.py`).

In [ ]:
display(Image(evaluate.OUT_DIR / 'data_sources.png'))

## Choosing the tree depth (5-fold cross-validation)

In [ ]:
display(Image(evaluate.OUT_DIR / 'cv_depth.png'))

## Confusion matrices
Rows = true level, columns = predicted level.

In [ ]:
for name in ['our_decision_tree', 'our_random_forest', 'sklearn_decision_tree', 'sklearn_random_forest']:
    display(Image(evaluate.OUT_DIR / f'confusion_{name}.png'))

## Feature importance (total Gini gain per feature)

In [ ]:
display(Image(evaluate.OUT_DIR / 'feature_importance.png'))

## One prediction, explained
The decision path of a single tree for a beginner on Everest Base Camp in January.

In [ ]:
from app.ml.dataset import load_treks, sample_for
from app.ml.features import FEATURES, LABELS, encode
from app.ml.tree import DecisionTree
from app.ml.train import load_dataset
from app.core.db import SessionLocal

with SessionLocal() as db:
    data = load_dataset(db)
tree = DecisionTree(max_depth=5).fit(data.X, data.y)
ebc = next(t for t in load_treks() if t.code == 'everest-base-camp')
x = encode(sample_for(ebc, month=1, age=30, fitness='medium', experience='none', health=False))
print('Prediction:', LABELS[tree.predict([x])[0]], tree.predict_proba([x])[0].round(2))
for c in tree.decision_path(x):
    print(f'  {FEATURES[c.feature]} {c.op} {c.threshold:g}')